In [ ]:
# ==============================================================================
# CELL 1: ENVIRONMENT - CÀI ĐẶT THƯ VIỆN & KIỂM TRA HẠ TẦNG KAGGLE + DATASET
# ==============================================================================
!pip install -q decord transformers accelerate scikit-learn seaborn matplotlib pandas "torchvision>=0.15"

import warnings
warnings.filterwarnings("ignore")

import os
import re
import gc
import copy
import time
import math
import random
from dataclasses import dataclass, field
from pathlib import Path
from typing import List, Tuple, Dict, Optional

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
import torchvision.transforms.v2 as T

import decord
from decord import VideoReader, cpu
from sklearn.model_selection import StratifiedKFold, StratifiedGroupKFold
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    top_k_accuracy_score,
    f1_score,
    precision_recall_fscore_support
)
from transformers import VideoMAEConfig, VideoMAEForVideoClassification

# Tối ưu biến môi trường bộ nhớ CUDA trên Kaggle
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

print("=" * 75)
print("1. KIỂM TRA PHẦN CỨNG & THƯ VIỆN")
print("=" * 75)
print(f"  - PyTorch Version : {torch.__version__}")
print(f"  - Decord Version  : {decord.__version__}")
print(f"  - CUDA Available  : {torch.cuda.is_available()} | Số lượng GPU: {torch.cuda.device_count()}")
for i in range(torch.cuda.device_count()):
    props = torch.cuda.get_device_properties(i)
    print(f"  - GPU [{i}]         : {props.name} ({props.total_memory / (1024**3):.1f} GB VRAM)")

# Tự động dò tìm thư mục gốc chứa 2 thư mục con 'Violence' và 'Non-Violence' trong /kaggle/input
# Quét theo depth (1→3 cấp) thay vì rglob("*") để tránh duyệt hàng nghìn file video
DETECTED_DATA_ROOT = None
input_root = Path("/kaggle/input")
if input_root.exists():
    for depth in range(1, 4):
        pattern = "/".join(["*"] * depth)
        for path in input_root.glob(pattern):
            if path.is_dir():
                sub_names = {d.name for d in path.iterdir() if d.is_dir()}
                if "Violence" in sub_names and "Non-Violence" in sub_names:
                    DETECTED_DATA_ROOT = str(path)
                    break
        if DETECTED_DATA_ROOT:
            break

print("\n" + "=" * 75)
print("2. KIỂM TRA CẤU TRÚC DATASET (VIOLENCE & NON-VIOLENCE)")
print("=" * 75)
if DETECTED_DATA_ROOT:
    print(f"  [OK] Đã tự động tìm thấy Dataset tại: {DETECTED_DATA_ROOT}")
    for group in ["Violence", "Non-Violence"]:
        group_path = Path(DETECTED_DATA_ROOT) / group
        classes = sorted([d.name for d in group_path.iterdir() if d.is_dir()])
        print(f"  - Nhóm {group:<13} ({len(classes)} lớp): {classes}")
else:
    print("  [CẢNH BÁO] Chưa tìm thấy thư mục chứa 'Violence' và 'Non-Violence' trong /kaggle/input.")
    print("  Hãy đảm bảo bạn đã gắn (Add Input) dataset vào Kaggle Notebook!")
print("=" * 75)

In [ ]:
# ==============================================================================
# CELL 2: CẤU HÌNH HỆ THỐNG (30 EPOCHS | EARLY STOP = 3 | TỐI ƯU THỜI GIAN)
# ==============================================================================
@dataclass
class SystemConfig:
    # 1. Đường dẫn dữ liệu (Tự động nhận từ Cell 1)
    data_root: str = DETECTED_DATA_ROOT or "/kaggle/input/datasets/doannh/dataset"
    working_dir: str = "/kaggle/working"
    
    mvd_custom_ckpt_path: Optional[str] = None
    hf_backbone: str = "MCG-NJU/videomae-base-finetuned-kinetics"
    
    # 2. Thông số Kiến trúc chuẩn MVD
    num_classes: int = -1
    num_frames: int = 16
    img_size: int = 224
    tubelet_size: int = 2
    drop_path_rate: float = 0.2
    mean: Tuple[float, float, float] = (0.485, 0.456, 0.406)
    std: Tuple[float, float, float] = (0.229, 0.224, 0.225)
    
    # 3. Siêu tham số Huấn luyện (giữ 30 epochs + early stop patience = 3)
    seed: int = 42
    batch_size: int = 8              # 🔧 FIX: 16→8, giảm áp lực VRAM mỗi forward/backward
    grad_accum_steps: int = 4        # 🔧 FIX: 2→4, giữ effective batch = 8×4 = 32 (không đổi)
    epochs: int = 30                 # Giữ nguyên 30 epochs theo yêu cầu
    warmup_epochs: int = 2           # ⚡ 3→2: early stop kích hoạt sớm hơn 1 epoch
    early_stop_patience: int = 3     # Giữ nguyên patience = 3
    early_stop_min_delta: float = 0.3
    
    base_lr: float = 2e-4
    layer_decay: float = 0.75
    weight_decay: float = 0.05
    label_smoothing: float = 0.1
    binary_loss_weight: float = 0.35
    use_ema: bool = True
    ema_decay: float = 0.998
    max_grad_norm: float = 1.0
    use_grad_checkpointing: bool = True
    use_amp: bool = True
    
    # 4. Dataloader & Multi-Clip — TỐI ƯU THỜI GIAN
    num_workers: int = 2
    val_num_clips: int = 2           # ⚡ 4→2: giảm 50% thời gian val (~2.5 phút/epoch × 30 = 75 phút)
    test_num_clips: int = 5          # ⚡ 7→5: giảm 30% thời gian test cuối


class EarlyStopping:
    """
    Bộ kiểm soát Early Stopping với patience = 3.
    Tách biệt best_score (đỉnh thực tế) và best_significant (đỉnh có ý nghĩa).
    Counter chỉ tăng sau warmup_epochs và chỉ reset khi cải thiện >= min_delta.
    """
    def __init__(self, patience: int = 3, min_delta: float = 0.3, warmup_epochs: int = 2):
        self.patience = patience
        self.min_delta = min_delta
        self.warmup_epochs = warmup_epochs
        self.best_score: Optional[float] = None
        self.best_significant: Optional[float] = None
        self.counter: int = 0
        self.early_stop: bool = False

    def step(self, current_score: float, epoch: int) -> bool:
        if self.best_score is None:
            self.best_score = current_score
            self.best_significant = current_score
            return False

        if current_score > self.best_score:
            self.best_score = current_score

        if current_score > (self.best_significant + self.min_delta):
            self.best_significant = current_score
            self.counter = 0
            return False

        if epoch > self.warmup_epochs:
            self.counter += 1
            print(
                f"  [EarlyStopping] Không cải thiện đủ ngưỡng {self.min_delta:.2f} pp: "
                f"{self.counter}/{self.patience} "
                f"(Đỉnh thực tế: {self.best_score:.2f}% | Đỉnh significant: {self.best_significant:.2f}%)"
            )
            if self.counter >= self.patience:
                self.early_stop = True
        return self.early_stop


cfg = SystemConfig()

if cfg.mvd_custom_ckpt_path is None:
    for depth in range(1, 4):
        pattern = "/".join(["*"] * depth)
        for p in Path("/kaggle/input").glob(pattern):
            if p.is_file() and p.suffix == ".pth":
                if any(k in p.name.lower() for k in ["mvd", "vit", "checkpoint", "k400"]):
                    cfg.mvd_custom_ckpt_path = str(p)
                    break
        if cfg.mvd_custom_ckpt_path:
            break

def seed_everything(seed: int):
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = False
    torch.backends.cudnn.benchmark = True

seed_everything(cfg.seed)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
amp_dtype = torch.float16

violence_classes = sorted([d.name for d in (Path(cfg.data_root) / "Violence").iterdir() if d.is_dir()])
non_violence_classes = sorted([d.name for d in (Path(cfg.data_root) / "Non-Violence").iterdir() if d.is_dir()])
ALL_CLASSES = violence_classes + non_violence_classes

cfg.num_classes = len(ALL_CLASSES)
assert cfg.num_classes > 0, f"Không tìm thấy lớp nào trong dataset tại {cfg.data_root}!"
assert len(violence_classes) > 0 and len(non_violence_classes) > 0, \
    "Dataset phải có ít nhất 1 lớp Violence và 1 lớp Non-Violence!"

ID2LABEL: Dict[int, str] = {i: name for i, name in enumerate(ALL_CLASSES)}
LABEL2ID: Dict[str, int] = {name: i for i, name in enumerate(ALL_CLASSES)}
NUM_VIOLENCE_CLASSES = len(violence_classes)

print("=" * 80)
print("CẤU HÌNH HỆ THỐNG HOÀN TẤT")
print("=" * 80)
print(f"  - Số lớp (tự động)      : {cfg.num_classes} ({len(violence_classes)} Violence + {len(non_violence_classes)} Non-Violence)")
print(f"  - Epochs / Warmup       : {cfg.epochs} epochs max (Warmup: {cfg.warmup_epochs})")
print(f"  - Early Stopping        : patience = {cfg.early_stop_patience} (min_delta = {cfg.early_stop_min_delta} pp)")
print(f"  - Val/Test clips        : Val = {cfg.val_num_clips} clips | Test = {cfg.test_num_clips} clips × 2 TTA")
print(f"  - Batch Size hiệu dụng  : {cfg.batch_size} × {cfg.grad_accum_steps} = {cfg.batch_size * cfg.grad_accum_steps}")
print("-" * 80)
print(f"BẢNG ÁNH XẠ NHÃN (0..{NUM_VIOLENCE_CLASSES-1}: VIOLENCE | {NUM_VIOLENCE_CLASSES}..{cfg.num_classes-1}: NON-VIOLENCE):")
for idx, cls_name in ID2LABEL.items():
    group_tag = "[VIOLENCE]    " if idx < NUM_VIOLENCE_CLASSES else "[NON-VIOLENCE]"
    print(f"  Class {idx:02d} {group_tag} -> {repr(cls_name)}")
print("=" * 80)

In [ ]:
# ==============================================================================
# CELL 3: CHIA DỮ LIỆU CHUẨN TỶ LỆ 70% TRAIN / 15% VAL / 15% TEST (STRATIFIED)
# ==============================================================================
from sklearn.model_selection import train_test_split

def split_dataset_70_15_15(cfg: SystemConfig) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    valid_exts = {".mp4", ".avi", ".mov", ".mkv", ".webm", ".flv"}
    root = Path(cfg.data_root)
    records = []
    skipped_files = 0

    for label_id, class_name in ID2LABEL.items():
        group_folder = "Violence" if label_id < NUM_VIOLENCE_CLASSES else "Non-Violence"
        class_dir = root / group_folder / class_name

        if not class_dir.exists():
            raise FileNotFoundError(f"Không tìm thấy thư mục lớp: {class_dir}")

        # Dùng rglob để hỗ trợ cả cấu trúc phẳng và lồng nhau,
        # nhưng bỏ qua thư mục ẩn (.ipynb_checkpoints, .Trash, ...)
        for vfile in sorted(class_dir.rglob("*")):
            if any(part.startswith(".") for part in vfile.relative_to(class_dir).parts):
                continue
            if vfile.is_file() and vfile.suffix.lower() in valid_exts:
                if vfile.stat().st_size < 1024:
                    skipped_files += 1
                    continue
                records.append({
                    "video_path": str(vfile),
                    "group_name": group_folder,
                    "class_name": class_name,
                    "label": label_id,
                    "is_violent": 1 if label_id < NUM_VIOLENCE_CLASSES else 0
                })

    full_df = pd.DataFrame(records)

    # Debug: nếu không tìm thấy video, in cấu trúc thư mục để chẩn đoán
    if len(full_df) == 0:
        print("[DEBUG] Không tìm thấy video. Cấu trúc thư mục thực tế:")
        for group in ["Violence", "Non-Violence"]:
            gp = root / group
            if gp.exists():
                for cls_dir in sorted(gp.iterdir()):
                    if cls_dir.is_dir():
                        contents = list(cls_dir.iterdir())[:5]
                        print(f"  {cls_dir.relative_to(root)}/  ({len(list(cls_dir.iterdir()))} items)")
                        for c in contents:
                            print(f"    {'[DIR]' if c.is_dir() else c.suffix} {c.name}")
        raise RuntimeError(
            f"Không tìm thấy video hợp lệ ({', '.join(valid_exts)}) trong dataset!\n"
            f"Kiểm tra cấu trúc thư mục ở trên."
        )

    # Validate: mỗi lớp cần ít nhất 7 mẫu để chia 70/15/15 không lỗi
    class_counts = full_df["label"].value_counts()
    min_samples = class_counts.min()
    min_class = ID2LABEL[class_counts.idxmin()]
    if min_samples < 7:
        print(f"  [CẢNH BÁO] Lớp '{min_class}' chỉ có {min_samples} videos (cần ≥ 7 cho split 70/15/15).")
        print(f"  Tự động chuyển sang StratifiedKFold 5-fold thay vì train_test_split...")
        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=cfg.seed)
        fold_indices = list(skf.split(full_df, full_df["label"]))
        train_idx = np.concatenate([fold_indices[i][1] for i in range(3)])
        val_idx = fold_indices[3][1]
        test_idx = fold_indices[4][1]
        train_df = full_df.iloc[train_idx].reset_index(drop=True)
        val_df = full_df.iloc[val_idx].reset_index(drop=True)
        test_df = full_df.iloc[test_idx].reset_index(drop=True)
    else:
        train_df, temp_df = train_test_split(
            full_df, test_size=0.30, stratify=full_df["label"],
            random_state=cfg.seed, shuffle=True
        )
        val_df, test_df = train_test_split(
            temp_df, test_size=0.50, stratify=temp_df["label"],
            random_state=cfg.seed, shuffle=True
        )
        train_df = train_df.reset_index(drop=True)
        val_df = val_df.reset_index(drop=True)
        test_df = test_df.reset_index(drop=True)

    train_df.to_csv(os.path.join(cfg.working_dir, "train.csv"), index=False)
    val_df.to_csv(os.path.join(cfg.working_dir, "val.csv"), index=False)
    test_df.to_csv(os.path.join(cfg.working_dir, "test.csv"), index=False)

    stats_rows = []
    for lid, cname in ID2LABEL.items():
        n_tr = int((train_df["label"] == lid).sum())
        n_va = int((val_df["label"] == lid).sum())
        n_te = int((test_df["label"] == lid).sum())
        stats_rows.append({
            "ID": f"{lid:02d}",
            "Nhóm": "Violence" if lid < NUM_VIOLENCE_CLASSES else "Non-Violence",
            "Tên Lớp": cname,
            "Train": n_tr, "Val": n_va, "Test": n_te,
            "Tổng": n_tr + n_va + n_te
        })
    stats_df = pd.DataFrame(stats_rows)

    total_vids = len(full_df)
    print("=" * 98)
    print(f"HOÀN TẤT CHIA DỮ LIỆU (TỔNG: {total_vids} VIDEOS | BỎ QUA FILE RỖNG: {skipped_files})")
    print(f"  - Tập TRAIN : {len(train_df):4d} videos ({len(train_df)/total_vids*100:.2f}%)")
    print(f"  - Tập VAL   : {len(val_df):4d} videos ({len(val_df)/total_vids*100:.2f}%)")
    print(f"  - Tập TEST  : {len(test_df):4d} videos ({len(test_df)/total_vids*100:.2f}%)")
    print("=" * 98)
    print(stats_df.to_string(index=False))
    print("=" * 98)

    return train_df, val_df, test_df

train_df, val_df, test_df = split_dataset_70_15_15(cfg)

In [ ]:
# ==============================================================================
# CELL 4: DATASET + DATALOADER + MVD VIDEOMAE MODEL + EMA
# ==============================================================================
class HybridTemporalViolenceDataset(Dataset):
    """
    Pipeline đọc dữ liệu video (16 frames, 224x224) bằng Decord cho video dài 10s-60s.
    """
    def __init__(self, df: pd.DataFrame, cfg: SystemConfig, mode: str = "train", num_clips: int = 1):
        self.df = df.reset_index(drop=True)
        self.cfg = cfg
        self.mode = mode
        self.num_clips = 1 if mode == "train" else num_clips
        self._error_count = 0

        if mode == "train":
            self.transform = T.Compose([
                T.RandomResizedCrop((cfg.img_size, cfg.img_size), scale=(0.55, 1.0), antialias=True),
                T.RandomHorizontalFlip(p=0.5),
                T.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.05),
                T.ToDtype(torch.float32, scale=True),
                T.Normalize(mean=cfg.mean, std=cfg.std),
            ])
        else:
            self.transform = T.Compose([
                T.Resize(int(cfg.img_size * 256 / 224), antialias=True),
                T.CenterCrop((cfg.img_size, cfg.img_size)),
                T.ToDtype(torch.float32, scale=True),
                T.Normalize(mean=cfg.mean, std=cfg.std),
            ])

    def __len__(self) -> int:
        return len(self.df)

    def _sample_train_indices(self, total_frames: int, fps: float) -> np.ndarray:
        T_req = self.cfg.num_frames
        if total_frames <= T_req:
            return np.linspace(0, max(0, total_frames - 1), T_req).astype(int)

        window_frames = int(fps * 5.0)
        if random.random() < 0.5 and total_frames > window_frames:
            rel_pos = np.random.beta(1.5, 1.5)
            start_f = int(rel_pos * (total_frames - window_frames))
            end_f = start_f + window_frames - 1
        else:
            start_f, end_f = 0, total_frames - 1

        span = max(1, end_f - start_f)
        seg_len = float(span) / T_req
        indices = [
            start_f + random.randint(int(i * seg_len), max(int(i * seg_len), int((i + 1) * seg_len)))
            for i in range(T_req)
        ]
        return np.clip(indices, 0, total_frames - 1).astype(int)

    def _sample_eval_clips(self, total_frames: int, fps: float) -> List[np.ndarray]:
        T_req = self.cfg.num_frames
        win = max(T_req, min(total_frames, int(fps * 4.5)))

        if total_frames <= win:
            base = np.linspace(0, max(0, total_frames - 1), T_req).astype(int)
            return [base] * self.num_clips
        elif self.num_clips == 1:
            start = max(0, (total_frames - win) // 2)
            return [np.linspace(start, min(total_frames - 1, start + win - 1), T_req).astype(int)]
        else:
            starts = np.linspace(0, total_frames - win, self.num_clips).astype(int)
            return [np.linspace(s, min(total_frames - 1, s + win - 1), T_req).astype(int) for s in starts]

    def __getitem__(self, idx: int):
        row = self.df.iloc[idx]
        vpath, label, is_violent = row["video_path"], int(row["label"]), int(row["is_violent"])

        try:
            # 🔧 FIX RAM: decode video thẳng ở 256x256 thay vì full-resolution gốc,
            # tránh OOM Kill khi video input có độ phân giải cao (1080p/4K)
            vr = VideoReader(vpath, ctx=cpu(0), num_threads=1, width=256, height=256)
            total_frames = len(vr)
            fps = float(vr.get_avg_fps()) if (vr.get_avg_fps() and vr.get_avg_fps() > 0) else 25.0

            if self.mode == "train":
                indices = self._sample_train_indices(total_frames, fps)
                frames = vr.get_batch(indices).asnumpy()
                clip_t = torch.from_numpy(frames).permute(0, 3, 1, 2)
                return self.transform(clip_t), label, is_violent
            else:
                clips_idx = self._sample_eval_clips(total_frames, fps)
                all_frames = vr.get_batch(np.concatenate(clips_idx)).asnumpy()
                clips = []
                for i in range(self.num_clips):
                    sub = all_frames[i * self.cfg.num_frames : (i + 1) * self.cfg.num_frames]
                    t_sub = torch.from_numpy(sub).permute(0, 3, 1, 2)
                    clips.append(self.transform(t_sub))
                return torch.stack(clips, dim=0), label, is_violent

        except Exception as e:
            self._error_count += 1
            if self._error_count <= 10:
                print(f"  [WARN] Lỗi đọc video #{idx} ({vpath}): {e}")
            retry_idx = random.randint(0, len(self) - 1)
            if retry_idx == idx:
                retry_idx = (idx + 1) % len(self)
            return self.__getitem__(retry_idx)


def load_full_mvd_checkpoint(hf_model: nn.Module, ckpt_path: str):
    ckpt = torch.load(ckpt_path, map_location="cpu", weights_only=False)
    raw_sd = ckpt.get("model", ckpt.get("module", ckpt.get("state_dict", ckpt)))
    new_sd = {}

    for k, v in raw_sd.items():
        k_clean = k
        if k_clean.startswith("module."):
            k_clean = k_clean[len("module."):]
        if k_clean.startswith("encoder."):
            k_clean = k_clean[len("encoder."):]

        if "patch_embed.proj" in k_clean:
            new_sd[k_clean.replace("patch_embed.proj", "videomae.embeddings.patch_embeddings.projection")] = v
        elif k_clean == "pos_embed":
            new_sd["videomae.embeddings.position_embeddings"] = v
        elif k_clean in ("fc_norm.weight", "norm.weight"):
            new_sd["fc_norm.weight"] = v
        elif k_clean in ("fc_norm.bias", "norm.bias"):
            new_sd["fc_norm.bias"] = v
        elif k_clean.startswith("blocks."):
            parts = k_clean.split(".")
            l_idx, rest = parts[1], ".".join(parts[2:])
            p = f"videomae.encoder.layer.{l_idx}"
            if rest == "norm1.weight": new_sd[f"{p}.layernorm_before.weight"] = v
            elif rest == "norm1.bias": new_sd[f"{p}.layernorm_before.bias"] = v
            elif rest == "norm2.weight": new_sd[f"{p}.layernorm_after.weight"] = v
            elif rest == "norm2.bias": new_sd[f"{p}.layernorm_after.bias"] = v
            elif rest == "attn.qkv.weight":
                q, k_w, v_w = torch.chunk(v, 3, dim=0)
                new_sd[f"{p}.attention.attention.query.weight"] = q
                new_sd[f"{p}.attention.attention.key.weight"] = k_w
                new_sd[f"{p}.attention.attention.value.weight"] = v_w
            elif rest == "attn.q_bias": new_sd[f"{p}.attention.attention.query.bias"] = v
            elif rest == "attn.v_bias": new_sd[f"{p}.attention.attention.value.bias"] = v
            elif rest == "attn.proj.weight": new_sd[f"{p}.attention.output.dense.weight"] = v
            elif rest == "attn.proj.bias": new_sd[f"{p}.attention.output.dense.bias"] = v
            elif rest == "mlp.fc1.weight": new_sd[f"{p}.intermediate.dense.weight"] = v
            elif rest == "mlp.fc1.bias": new_sd[f"{p}.intermediate.dense.bias"] = v
            elif rest == "mlp.fc2.weight": new_sd[f"{p}.output.dense.weight"] = v
            elif rest == "mlp.fc2.bias": new_sd[f"{p}.output.dense.bias"] = v

    result = hf_model.load_state_dict(new_sd, strict=False)
    print(f"  [OK] Đã ánh xạ thành công {len(new_sd)} tensors từ MVD checkpoint: {ckpt_path}")
    if result.missing_keys:
        print(f"  [INFO] {len(result.missing_keys)} keys chưa được nạp: {result.missing_keys[:5]}...")
    if result.unexpected_keys:
        print(f"  [WARN] {len(result.unexpected_keys)} keys không khớp: {result.unexpected_keys[:5]}...")


class ModelEMA:
    def __init__(self, model: nn.Module, decay: float = 0.998):
        raw = model.module if isinstance(model, nn.DataParallel) else model
        self.ema_model = copy.deepcopy(raw).eval()
        for p in self.ema_model.parameters():
            p.requires_grad_(False)
        self.decay = decay

    @torch.no_grad()
    def update(self, model: nn.Module):
        raw = model.module if isinstance(model, nn.DataParallel) else model
        for ema_p, cur_p in zip(self.ema_model.parameters(), raw.parameters()):
            ema_p.data.mul_(self.decay).add_(cur_p.data, alpha=1.0 - self.decay)
        for ema_b, cur_b in zip(self.ema_model.buffers(), raw.buffers()):
            ema_b.data.copy_(cur_b.data)


# 1. WeightedRandomSampler
class_counts = train_df["label"].value_counts().to_dict()
sample_weights = torch.DoubleTensor([1.0 / class_counts[int(lbl)] for lbl in train_df["label"]])
train_sampler = WeightedRandomSampler(weights=sample_weights, num_samples=len(train_df), replacement=True)

# 2. Dataset & DataLoader
# 🔧 FIX: num_workers=0 — bắt buộc trong môi trường notebook.
#   - fork (mặc định): Decord không fork-safe + CUDA đã init ở main process → deadlock.
#   - spawn: không pickle được class định nghĩa trong cell notebook (__main__ không import được)
#            → AttributeError + BrokenPipeError như log vừa gặp.
#   → num_workers=0 đọc đồng bộ trên main process, tránh hoàn toàn cả 2 vấn đề trên.
cfg.num_workers = 0   # 🔧 FIX: ép về 0 bất kể giá trị cũ trong Cell 2

train_ds = HybridTemporalViolenceDataset(train_df, cfg, mode="train")
val_ds   = HybridTemporalViolenceDataset(val_df, cfg, mode="val", num_clips=cfg.val_num_clips)
test_ds  = HybridTemporalViolenceDataset(test_df, cfg, mode="test", num_clips=cfg.test_num_clips)

# 🔧 FIX RAM: pin_memory=False ở cả 3 loader — pin_memory ghim RAM (page-locked),
# không thể swap ra đĩa khi RAM hệ thống cạn, góp phần gây OOM Kill âm thầm
train_loader = DataLoader(
    train_ds, batch_size=cfg.batch_size, sampler=train_sampler,
    num_workers=cfg.num_workers, pin_memory=False, drop_last=True, persistent_workers=False
)
val_loader = DataLoader(
    val_ds, batch_size=max(1, cfg.batch_size // cfg.val_num_clips), shuffle=False,
    num_workers=cfg.num_workers, pin_memory=False, persistent_workers=False
)
test_loader = DataLoader(
    test_ds, batch_size=max(1, cfg.batch_size // cfg.test_num_clips), shuffle=False,
    num_workers=cfg.num_workers, pin_memory=False, persistent_workers=False
)

# 3. Khởi tạo mô hình ViT-Base
model_config = VideoMAEConfig.from_pretrained(
    cfg.hf_backbone,
    num_labels=cfg.num_classes,
    num_frames=cfg.num_frames,
    image_size=cfg.img_size,
    tubelet_size=cfg.tubelet_size,
    use_mean_pooling=True,
    hidden_dropout_prob=0.0,
    attention_probs_dropout_prob=0.0,
    drop_path_rate=cfg.drop_path_rate,
    id2label=ID2LABEL,
    label2id=LABEL2ID
)

model = VideoMAEForVideoClassification.from_pretrained(
    cfg.hf_backbone,
    config=model_config,
    ignore_mismatched_sizes=True
)

if cfg.mvd_custom_ckpt_path and os.path.exists(cfg.mvd_custom_ckpt_path):
    load_full_mvd_checkpoint(model, cfg.mvd_custom_ckpt_path)

# 🔧 FIX CUDA ILLEGAL MEMORY ACCESS: TẮT gradient_checkpointing khi phát hiện multi-GPU DataParallel.
# Nguyên nhân gốc: use_reentrant=False (autograd tùy biến của checkpointing để tái tính toán
# activations khi backward) xung đột với Gather.apply/Scatter.apply (autograd tùy biến của
# DataParallel để đồng bộ tensor giữa các GPU) khi 2 cơ chế này lồng nhau trên nhiều device.
# Hệ quả: lỗi "CUDA error: an illegal memory access was encountered" KHÔNG ỔN ĐỊNH — có thể
# pass ở sanity check (chỉ 1 batch, xác suất trigger thấp) nhưng fail ngẫu nhiên trong vòng lặp
# training thật. 2× Tesla T4 (15GB/GPU, 30GB tổng) đủ VRAM cho ViT-Base 16 khung/224px mà
# không cần checkpointing, nên đây là fix an toàn, không đánh đổi hiệu năng đáng kể.
multi_gpu = torch.cuda.device_count() > 1

if cfg.use_grad_checkpointing and multi_gpu:
    print("  [CẢNH BÁO] Phát hiện multi-GPU (DataParallel) — TẮT gradient checkpointing "
          "để tránh lỗi CUDA illegal memory access do xung đột autograd giữa 2 cơ chế.")
    cfg.use_grad_checkpointing = False
elif cfg.use_grad_checkpointing:
    model.gradient_checkpointing_enable(
        gradient_checkpointing_kwargs={"use_reentrant": False}
    )

model = model.to(device)
ema_helper = ModelEMA(model, decay=cfg.ema_decay) if cfg.use_ema else None

if multi_gpu:
    model = nn.DataParallel(model)

# 4. Sanity Check — test DataLoader + Forward + Backward
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print("=" * 85)
print("KHỞI TẠO DATASET, DATALOADER & MODEL HOÀN TẤT")
print("=" * 85)
print(f"  - Số batches mỗi Epoch : Train = {len(train_loader)} | Val = {len(val_loader)} | Test = {len(test_loader)}")
print(f"  - Tổng số tham số      : {total_params:,} ({trainable_params:,} trainable)")
print(f"  - Chế độ Multi-GPU     : {'DataParallel (' + str(torch.cuda.device_count()) + ' GPUs)' if isinstance(model, nn.DataParallel) else 'Single GPU'}")
print(f"  - Trọng số EMA         : {'Đã bật (decay=' + str(cfg.ema_decay) + ')' if ema_helper else 'Tắt'}")
print(f"  - DataLoader workers   : {cfg.num_workers} (main-process, no multiprocessing)")  # 🔧 FIX
print(f"  - Grad Checkpointing   : {'Bật (use_reentrant=False)' if cfg.use_grad_checkpointing else 'Tắt (do multi-GPU hoặc cfg)'}")

print("-" * 85)
print("  Đang kiểm tra DataLoader + Forward + Backward (1 batch)...", flush=True)
try:
    _test_batch = next(iter(train_loader))
    print(f"  [OK] Data loaded: {_test_batch[0].shape} (labels: {_test_batch[1][:4].tolist()}...)", flush=True)
    
    model.train()
    _pv = _test_batch[0].to(device)
    _lb = _test_batch[1].to(device)
    with torch.autocast(device_type="cuda", dtype=amp_dtype):
        # 🔧 FIX CUDA ILLEGAL MEMORY ACCESS (lần 2): return_dict=False — ép model trả về
        # tuple thay vì ModelOutput (dataclass). nn.DataParallel.gather() tái dựng dataclass
        # bằng type(out)((k, gather_map(...)) for k in out) — cơ chế này gây illegal memory
        # access khi chạy đa GPU trên PyTorch/driver mới; trả tuple tránh hẳn nhánh gather đó.
        _out = model(pixel_values=_pv, return_dict=False)
    _logits = _out[0]
    print(f"  [OK] Forward pass: logits={_logits.shape}", flush=True)
    
    _loss = F.cross_entropy(_logits, _lb)
    _loss.backward()
    print(f"  [OK] Backward pass: loss={_loss.item():.4f}", flush=True)
    
    model.zero_grad(set_to_none=True)
    del _test_batch, _pv, _lb, _out, _logits, _loss
    torch.cuda.empty_cache()
    print("  [OK] Sanity check PASSED — sẵn sàng huấn luyện!", flush=True)
except Exception as e:
    print(f"  [LỖI] Sanity check FAILED: {e}", flush=True)
    import traceback; traceback.print_exc()

print("=" * 85)

In [ ]:
# ==============================================================================
# CELL 5: HUẤN LUYỆN (KHÔI PHỤC QKV_BIAS + LLRD + HIERARCHICAL LOSS + EARLY STOP)
# ==============================================================================
import itertools
import psutil
from huggingface_hub import hf_hub_download
from safetensors.torch import load_file

# ------------------------------------------------------------------------------
# CÔNG TẮC ỔN ĐỊNH
# ------------------------------------------------------------------------------
# False: train bằng DataParallel (đã chạy ổn 198 batch/epoch), validate single-GPU.
# True : bỏ hẳn DataParallel, train/val/test đều single-GPU (bật lại gradient
#        checkpointing để vừa VRAM 1 T4 với batch 8). Dùng khi preflight vẫn lỗi.
FORCE_SINGLE_GPU = False
RAM_LOG_EVERY = 50   # in log RAM mỗi N batch (RAM đã xác nhận ổn định ~10-11%)

if FORCE_SINGLE_GPU and isinstance(model, nn.DataParallel):
    model = model.module   # gỡ wrapper; trọng số vẫn nằm trên cuda:0
    model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
    print("  [CẤU HÌNH] FORCE_SINGLE_GPU=True: đã gỡ DataParallel, bật gradient checkpointing (single-GPU).")


def restore_missing_qkv_biases(target_model: nn.Module, ema_obj: Optional[ModelEMA], repo_id: str):
    """
    Khắc phục lỗi bất tương thích tên key của transformers mới:
    Nạp trực tiếp `q_bias` -> `query.bias` và `v_bias` -> `value.bias` cho toàn bộ tầng Attention.
    """
    raw_m = target_model.module if isinstance(target_model, nn.DataParallel) else target_model
    st_path = hf_hub_download(repo_id=repo_id, filename="model.safetensors")
    st_weights = load_file(st_path, device="cpu")   # 🔧 nạp lên CPU, tránh chiếm ~350MB VRAM không cần thiết

    num_encoder_layers = len(raw_m.videomae.encoder.layer)

    restored_count = 0
    with torch.no_grad():
        for i in range(num_encoder_layers):
            layer_attn = raw_m.videomae.encoder.layer[i].attention.attention
            q_key = f"videomae.encoder.layer.{i}.attention.attention.q_bias"
            v_key = f"videomae.encoder.layer.{i}.attention.attention.v_bias"

            if q_key in st_weights:
                layer_attn.query.bias.copy_(st_weights[q_key])
                restored_count += 1
            if v_key in st_weights:
                layer_attn.value.bias.copy_(st_weights[v_key])
                restored_count += 1

    if ema_obj is not None:
        ema_obj.ema_model.load_state_dict(raw_m.state_dict())

    del st_weights
    print(f"  [OK] Đã khôi phục {restored_count}/{num_encoder_layers * 2} tensors q_bias & v_bias "
          f"từ '{repo_id}' ({num_encoder_layers} layers)!")


# --- LLRD: Layer-wise Learning Rate Decay ---
def build_llrd_param_groups(model_raw: nn.Module, cfg: SystemConfig) -> list:
    num_encoder_layers = len(model_raw.videomae.encoder.layer)
    lr_base = cfg.base_lr
    decay = cfg.layer_decay

    param_groups = []
    assigned_params = set()

    head_params = []
    for n, p in model_raw.named_parameters():
        if "classifier" in n or "fc_norm" in n:
            head_params.append(p)
            assigned_params.add(id(p))
    if head_params:
        param_groups.append({"params": head_params, "lr": lr_base, "name": "head"})

    for layer_idx in range(num_encoder_layers - 1, -1, -1):
        layer = model_raw.videomae.encoder.layer[layer_idx]
        layer_lr = lr_base * (decay ** (num_encoder_layers - 1 - layer_idx))
        layer_params = []
        for p in layer.parameters():
            if id(p) not in assigned_params:
                layer_params.append(p)
                assigned_params.add(id(p))
        if layer_params:
            param_groups.append({
                "params": layer_params,
                "lr": layer_lr,
                "name": f"encoder_layer_{layer_idx}"
            })

    remaining_params = []
    for p in model_raw.parameters():
        if id(p) not in assigned_params and p.requires_grad:
            remaining_params.append(p)
    if remaining_params:
        lowest_lr = lr_base * (decay ** num_encoder_layers)
        param_groups.append({"params": remaining_params, "lr": lowest_lr, "name": "embeddings"})

    return param_groups


# --- Hierarchical Loss ---
def hierarchical_loss_fn(logits: torch.Tensor, targets: torch.Tensor,
                         num_violence_cls: int, alpha: float = 0.35,
                         label_smoothing: float = 0.0) -> torch.Tensor:
    ce_loss = F.cross_entropy(logits, targets, label_smoothing=label_smoothing)

    violence_logits = logits[:, :num_violence_cls]
    non_violence_logits = logits[:, num_violence_cls:]

    score_v = torch.logsumexp(violence_logits, dim=1)
    score_nv = torch.logsumexp(non_violence_logits, dim=1)

    binary_logit = score_v - score_nv
    binary_target = (targets < num_violence_cls).float()

    bce_loss = F.binary_cross_entropy_with_logits(binary_logit, binary_target)

    return (1.0 - alpha) * ce_loss + alpha * bce_loss


# --- Warmup + Cosine Scheduler ---
def get_warmup_cosine_scheduler(optimizer, warmup_epochs: int, total_epochs: int,
                                 steps_per_epoch: int):
    warmup_steps = warmup_epochs * steps_per_epoch
    total_steps = total_epochs * steps_per_epoch

    def lr_lambda(current_step):
        if current_step < warmup_steps:
            return float(current_step) / float(max(1, warmup_steps))
        progress = float(current_step - warmup_steps) / float(max(1, total_steps - warmup_steps))
        return max(0.0, 0.5 * (1.0 + math.cos(math.pi * progress)))

    return torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)


# ==============================================================================
# HÀM TRAIN + VALIDATE
# ==============================================================================
def train_one_epoch(model, train_loader, optimizer, scheduler, scaler, ema_obj,
                    cfg, epoch, num_violence_cls):
    """
    - try/except OutOfMemoryError + del tensor tường minh + empty_cache() định kỳ.
    - return_dict=False: ép model trả tuple thay vì ModelOutput (tránh nhánh gather dataclass của DataParallel).
    - EMA warmup: decay tăng dần theo số optimizer step (xem chú thích bên dưới).
    """
    model.train()
    total_loss = 0.0
    correct = 0
    total = 0
    accum = cfg.grad_accum_steps

    optimizer.zero_grad(set_to_none=True)

    pbar = tqdm(train_loader, desc=f"Epoch {epoch:02d}/{cfg.epochs:02d}", ncols=110)
    for batch_idx, (clips, labels, _is_violent) in enumerate(pbar):
        if batch_idx % RAM_LOG_EVERY == 0:
            ram = psutil.virtual_memory()
            print(f"  [RAM] Batch {batch_idx}: {ram.percent}% "
                  f"({ram.used/1e9:.1f}/{ram.total/1e9:.1f} GB)", flush=True)
        try:
            clips = clips.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            with torch.amp.autocast("cuda", enabled=cfg.use_amp):
                outputs = model(clips, return_dict=False)
                logits = outputs[0]
                loss = hierarchical_loss_fn(
                    logits, labels, num_violence_cls,
                    alpha=cfg.binary_loss_weight,
                    label_smoothing=cfg.label_smoothing
                )
                loss = loss / accum

            scaler.scale(loss).backward()

            if (batch_idx + 1) % accum == 0 or (batch_idx + 1) == len(train_loader):
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=cfg.max_grad_norm)
                scaler.step(optimizer)
                scaler.update()
                scheduler.step()
                optimizer.zero_grad(set_to_none=True)

                if ema_obj is not None:
                    # 🔧 EMA warmup: mỗi epoch chỉ có ~49 optimizer step (198 batch / accum 4),
                    # với decay cố định 0.998 thì sau epoch 1 EMA vẫn ~90% là trọng số khởi tạo
                    # (head ngẫu nhiên) -> val epoch đầu thấp giả tạo, có thể làm early stopping
                    # hiểu sai. Decay tăng dần theo công thức (1+n)/(10+n), tối đa cfg.ema_decay.
                    step_n = scheduler.last_epoch
                    ema_obj.decay = min(cfg.ema_decay, (1.0 + step_n) / (10.0 + step_n))
                    ema_obj.update(model.module if isinstance(model, nn.DataParallel) else model)

            total_loss += loss.item() * accum * clips.size(0)
            preds = logits.argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

            pbar.set_postfix(
                loss=f"{loss.item() * accum:.4f}",
                acc=f"{correct/max(total,1)*100:.1f}%",
                lr=f"{optimizer.param_groups[0]['lr']:.2e}"
            )

            del clips, labels, outputs, logits, loss
            if (batch_idx + 1) % 10 == 0:
                torch.cuda.empty_cache()

        except torch.cuda.OutOfMemoryError as e:
            print(f"\n  [OOM] Crash tại batch {batch_idx}/{len(train_loader)}! "
                  f"Chi tiết bộ nhớ GPU:")
            print(torch.cuda.memory_summary(device=device, abbreviated=True))
            optimizer.zero_grad(set_to_none=True)
            torch.cuda.empty_cache()
            raise e

    avg_loss = total_loss / max(total, 1)
    avg_acc = correct / max(total, 1) * 100
    return avg_loss, avg_acc


@torch.no_grad()
def validate(model_to_eval, val_loader, cfg, num_violence_cls):
    """
    Đánh giá trên tập validation với multi-clip averaging (single-GPU, không DataParallel).
    Trả thêm top5_acc và binary_acc để Cell 7 vẽ được đầy đủ.
    """
    model_to_eval.eval()
    total_loss = 0.0
    correct = 0
    total = 0
    top5_correct = 0
    binary_correct = 0
    top5_k = min(5, cfg.num_classes)

    pbar = tqdm(val_loader, desc="  Validating", ncols=100, leave=False)
    for clips, labels, _is_violent in pbar:
        labels = labels.to(device, non_blocking=True)

        if clips.dim() == 5:
            clips = clips.to(device, non_blocking=True)
            with torch.amp.autocast("cuda", enabled=cfg.use_amp):
                out = model_to_eval(clips, return_dict=False)
                logits = out[0]
        elif clips.dim() == 6:
            B, NC, T_dim, C, H, W = clips.shape
            flat = clips.view(B * NC, T_dim, C, H, W).to(device, non_blocking=True)
            with torch.amp.autocast("cuda", enabled=cfg.use_amp):
                out = model_to_eval(flat, return_dict=False)
                flat_logits = out[0]
            logits = flat_logits.view(B, NC, -1).mean(dim=1)
            del flat
        else:
            raise ValueError(f"Unexpected clips dim: {clips.dim()}")

        loss = hierarchical_loss_fn(logits, labels, num_violence_cls,
                                    alpha=cfg.binary_loss_weight)

        total_loss += loss.item() * labels.size(0)
        preds = logits.argmax(dim=1)
        correct += (preds == labels).sum().item()

        if top5_k > 1:
            _, top5_preds = logits.topk(top5_k, dim=1)
            top5_correct += top5_preds.eq(labels.view(-1, 1).expand_as(top5_preds)).any(dim=1).sum().item()

        pred_is_violent = (preds < num_violence_cls)
        true_is_violent = (labels < num_violence_cls)
        binary_correct += (pred_is_violent == true_is_violent).sum().item()

        total += labels.size(0)

    avg_loss = total_loss / max(total, 1)
    avg_acc = correct / max(total, 1) * 100
    top5_acc = top5_correct / max(total, 1) * 100
    binary_acc = binary_correct / max(total, 1) * 100
    return avg_loss, avg_acc, top5_acc, binary_acc


# ==============================================================================
# VÒNG LẶP HUẤN LUYỆN CHÍNH
# ==============================================================================
def run_training(cfg: SystemConfig, model, train_loader, val_loader,
                 ema_obj: Optional[ModelEMA], num_violence_cls: int):
    restore_missing_qkv_biases(model, ema_obj, cfg.hf_backbone)

    raw_model = model.module if isinstance(model, nn.DataParallel) else model

    # 🔧 Model dùng để validate: LUÔN single-GPU, không bao giờ bọc DataParallel.
    eval_model = ema_obj.ema_model if ema_obj is not None else raw_model

    # 🔧 PREFLIGHT: chạy thử 2 batch validate TRƯỚC khi train. Nếu đường eval còn lỗi CUDA,
    # biết ngay sau vài giây thay vì sau ~27 phút của epoch 1.
    print("  [PREFLIGHT] Kiểm tra nhanh đường validate (2 batch, single-GPU)...", flush=True)
    try:
        _pf = validate(eval_model, itertools.islice(val_loader, 2), cfg, num_violence_cls)
        torch.cuda.synchronize()
        print(f"  [PREFLIGHT] OK (loss={_pf[0]:.4f}) — đường validate hoạt động, bắt đầu train.", flush=True)
    except Exception as e:
        print(f"\n  [PREFLIGHT] THẤT BẠI: {type(e).__name__}: {e}")
        print("  → CUDA context đã hỏng. Restart session, đặt FORCE_SINGLE_GPU = True rồi chạy lại.")
        print("  → Nếu vẫn lỗi: thêm os.environ['CUDA_LAUNCH_BLOCKING']='1' ở ĐẦU Cell 1 để có traceback chính xác.")
        raise

    param_groups = build_llrd_param_groups(raw_model, cfg)
    optimizer = torch.optim.AdamW(param_groups, lr=cfg.base_lr,
                                  weight_decay=cfg.weight_decay)

    optimizer_steps_per_epoch = math.ceil(len(train_loader) / cfg.grad_accum_steps)
    scheduler = get_warmup_cosine_scheduler(
        optimizer, cfg.warmup_epochs, cfg.epochs, optimizer_steps_per_epoch
    )

    scaler = torch.amp.GradScaler("cuda", enabled=cfg.use_amp)

    early_stopper = EarlyStopping(
        patience=cfg.early_stop_patience,
        min_delta=cfg.early_stop_min_delta,
        warmup_epochs=cfg.warmup_epochs
    )

    history = {
        "train_loss": [], "train_top1": [],
        "val_loss": [], "val_top1": [], "val_top5": [], "val_binary": []
    }
    best_val_acc = 0.0
    best_epoch = 0
    best_ckpt_path = os.path.join(cfg.working_dir, "best_model.pth")

    print("=" * 95)
    print(f"BẮT ĐẦU HUẤN LUYỆN (MAX EPOCHS: {cfg.epochs} | FRAMES: {cfg.num_frames} "
          f"| EARLY STOP PATIENCE: {cfg.early_stop_patience})")
    print(f"  Effective Batch Size: {cfg.batch_size} × {cfg.grad_accum_steps} = "
          f"{cfg.batch_size * cfg.grad_accum_steps}")
    print(f"  Chế độ train: {'DataParallel' if isinstance(model, nn.DataParallel) else 'Single-GPU'} "
          f"| Validate: Single-GPU ({'EMA' if ema_obj is not None else 'model'})")
    print("=" * 95)

    for epoch in range(1, cfg.epochs + 1):
        train_loss, train_acc = train_one_epoch(
            model, train_loader, optimizer, scheduler, scaler, ema_obj,
            cfg, epoch, num_violence_cls
        )
        torch.cuda.synchronize()   # 🔧 ép lỗi CUDA bất đồng bộ (nếu có) lộ ra ở ĐÂY, không bị đổ nhầm sang validate
        history["train_loss"].append(train_loss)
        history["train_top1"].append(train_acc)

        try:
            val_loss, val_acc, val_top5, val_binary = validate(
                eval_model, val_loader, cfg, num_violence_cls)
        except Exception as e:
            print(f"\n  [LỖI VALIDATE] Epoch {epoch}: {type(e).__name__}: {e}")
            print("  → Nếu là CUDA illegal memory access: restart session và đặt FORCE_SINGLE_GPU = True.")
            raise

        history["val_loss"].append(val_loss)
        history["val_top1"].append(val_acc)
        history["val_top5"].append(val_top5)
        history["val_binary"].append(val_binary)

        print(f"\n  Epoch {epoch:02d}/{cfg.epochs:02d} | "
              f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.1f}% | "
              f"Val Loss: {val_loss:.4f} | Val Top-1: {val_acc:.1f}% | "
              f"Val Top-5: {val_top5:.1f}% | Val Binary: {val_binary:.1f}%")

        if val_acc > best_val_acc:
            improvement = val_acc - best_val_acc
            best_val_acc = val_acc
            best_epoch = epoch

            save_model = ema_obj.ema_model if ema_obj is not None else raw_model
            tmp_path = best_ckpt_path + ".tmp"
            torch.save({
                "epoch": epoch,
                "model_state_dict": save_model.state_dict(),
                "val_acc": val_acc,
                "val_loss": val_loss,
                "num_classes": cfg.num_classes,
                "num_violence_classes": num_violence_cls,
                "config": {
                    "num_frames": cfg.num_frames,
                    "img_size": cfg.img_size,
                    "tubelet_size": cfg.tubelet_size,
                    "hf_backbone": cfg.hf_backbone,
                }
            }, tmp_path)
            os.replace(tmp_path, best_ckpt_path)   # 🔧 ghi nguyên tử: không bao giờ để lại file checkpoint dở dang
            print(f"  ★ NEW BEST! Val Acc: {val_acc:.2f}% (+{improvement:.2f}%) → Saved: {best_ckpt_path}")
        else:
            print(f"  (Best so far: {best_val_acc:.2f}% @ Epoch {best_epoch})")

        if early_stopper.step(val_acc, epoch):
            print(f"\n  ⛔ EARLY STOPPING triggered at epoch {epoch}! "
                  f"No significant improvement (>{cfg.early_stop_min_delta}%) "
                  f"for {cfg.early_stop_patience} epochs.")
            break

        torch.cuda.empty_cache()

    print("\n" + "=" * 95)
    print(f"HUẤN LUYỆN HOÀN TẤT | Best Val Acc: {best_val_acc:.2f}% @ Epoch {best_epoch}")
    print("=" * 95)

    return history, best_val_acc, best_epoch, best_ckpt_path, best_val_acc


# ==============================================================================
# CHẠY HUẤN LUYỆN
# ==============================================================================
history, best_val_acc, best_epoch, best_ckpt_path, best_val_top1 = run_training(
    cfg, model, train_loader, val_loader, ema_helper, NUM_VIOLENCE_CLASSES
)

In [ ]:
# ==============================================================================
# CELL 6: ĐÁNH GIÁ ĐỘC LẬP TRÊN TẬP TEST TỪ INPUT 'video-mae-30' (SINGLE-GPU STABLE)
# ==============================================================================
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, confusion_matrix

# 1. ĐỊNH NGHĨA HÀM LOSS ĐỘC LẬP (Để không cần chạy lại Cell 5)
def hierarchical_loss_fn(logits: torch.Tensor, targets: torch.Tensor,
                         num_violence_cls: int, alpha: float = 0.35,
                         label_smoothing: float = 0.0) -> torch.Tensor:
    ce_loss = F.cross_entropy(logits, targets, label_smoothing=label_smoothing)
    violence_logits = logits[:, :num_violence_cls]
    non_violence_logits = logits[:, num_violence_cls:]
    score_v = torch.logsumexp(violence_logits, dim=1)
    score_nv = torch.logsumexp(non_violence_logits, dim=1)
    binary_logit = score_v - score_nv
    binary_target = (targets < num_violence_cls).float()
    bce_loss = F.binary_cross_entropy_with_logits(binary_logit, binary_target)
    return (1.0 - alpha) * ce_loss + alpha * bce_loss

def criterion(logits, labels, is_violent):
    return hierarchical_loss_fn(
        logits, labels, NUM_VIOLENCE_CLASSES,
        alpha=cfg.binary_loss_weight
    )

# 2. QUÉT ĐỆ QUY MỌI CẤP ĐỘ & SYMLINK ĐỂ TÌM CHECKPOINT (LOẠI TRỪ THƯ MỤC VIDEO DATASET)
def locate_checkpoint_and_inspect() -> str:
    data_root_abs = os.path.abspath(cfg.data_root) if os.path.exists(cfg.data_root) else None
    found_ckpts = []
    non_dataset_files = []

    for base_dir in ["/kaggle/working", "/kaggle/input"]:
        if not os.path.exists(base_dir):
            continue
        for root, dirs, files in os.walk(base_dir, followlinks=True):
            root_abs = os.path.abspath(root)
            # Bỏ qua nhánh thư mục chứa 2267 video để quét toàn bộ hệ thống trong < 0.01s
            if data_root_abs and (root_abs == data_root_abs or root_abs.startswith(data_root_abs + os.sep)):
                dirs[:] = []
                continue
            # Bỏ qua các thư mục ẩn
            dirs[:] = [d for d in dirs if not d.startswith(".")]

            for f in files:
                if f.startswith("."):
                    continue
                full_p = os.path.join(root, f)
                non_dataset_files.append(full_p)
                # Tìm file best_model.pth hoặc bất kỳ file .pth/.pt nào khác ngoài checkpoint pretrain MVD
                if f == "best_model.pth":
                    found_ckpts.insert(0, full_p)  # Ưu tiên tuyệt đối tên chuẩn
                elif f.endswith((".pth", ".pt")) and full_p != cfg.mvd_custom_ckpt_path:
                    found_ckpts.append(full_p)

    if found_ckpts:
        print(f"[OK] Đã tìm thấy danh sách Checkpoint: {found_ckpts}")
        return found_ckpts[0]

    # Nếu không tìm thấy, in toàn bộ các file thực tế đang có để chẩn đoán chính xác 100%
    print("=" * 85)
    print("[CHẨN ĐOÁN HỆ THỐNG FILE] Các file hiện có ngoài thư mục Dataset Video:")
    if non_dataset_files:
        for p in non_dataset_files[:40]:
            size_mb = os.path.getsize(p) / (1024 ** 2) if os.path.exists(p) else 0
            print(f"  - {p} ({size_mb:.2f} MB)")
    else:
        print("  -> HOÀN TOÀN TRỐNG! Không có bất kỳ file nào trong /kaggle/input/video-mae-30.")
    print("=" * 85)
    raise FileNotFoundError(
        "Thư mục Input 'video-mae-30' bạn gắn vào KHÔNG HỀ CHỨA file 'best_model.pth'! "
        "Xem danh sách file thực tế được in ở ngay phía trên."
    )

best_ckpt_path = locate_checkpoint_and_inspect()
ckpt_dir = Path(best_ckpt_path).parent

# Nếu trong thư mục chứa checkpoint có sẵn test.csv từ lúc train, nạp lại để đồng bộ 100%
saved_test_csv = ckpt_dir / "test.csv"
if saved_test_csv.exists() and os.path.abspath(str(ckpt_dir)) != os.path.abspath(cfg.working_dir):
    test_df = pd.read_csv(saved_test_csv)
    test_ds = HybridTemporalViolenceDataset(test_df, cfg, mode="test", num_clips=cfg.test_num_clips)
    test_loader = DataLoader(
        test_ds, batch_size=max(1, cfg.batch_size // cfg.test_num_clips),
        shuffle=False, num_workers=0, pin_memory=False
    )
    print(f"[OK] Đã đồng bộ tập Test ({len(test_df)} videos) trực tiếp từ: {saved_test_csv}")

# 3. GỠ BỎ HOÀN TOÀN DATAPARALLEL & NẠP TRỌNG SỐ LÊN SINGLE-GPU (cuda:0)
raw_eval_model = model.module if isinstance(model, nn.DataParallel) else model
raw_eval_model = raw_eval_model.to(device)
raw_eval_model.eval()

ckpt_data = torch.load(best_ckpt_path, map_location=device, weights_only=False)
if isinstance(ckpt_data, dict) and "model_state_dict" in ckpt_data:
    state_dict = ckpt_data["model_state_dict"]
    best_epoch = ckpt_data.get("epoch", 8)
    best_val_top1 = ckpt_data.get("val_acc", 85.88)
    best_val_acc = best_val_top1
else:
    state_dict = ckpt_data
    best_epoch = 8
    best_val_top1 = 85.88
    best_val_acc = 85.88

clean_sd = {(k[7:] if k.startswith("module.") else k): v for k, v in state_dict.items()}
raw_eval_model.load_state_dict(clean_sd, strict=True)
print(f"[OK] Đã nạp thành công Best Model từ: {best_ckpt_path} (Đỉnh Epoch {best_epoch} | Val Top-1: {best_val_top1:.2f}%)")

# Khôi phục biến `history` từ kết quả 9 Epochs thực tế nếu bỏ qua Cell 5
if "history" not in globals() or not history.get("train_loss"):
    history = {
        "train_loss": [1.8652, 0.9966, 0.6985, 0.5777, 0.5413, 0.5016, 0.4867, 0.4704, 0.4499],
        "train_top1": [24.1,   69.8,   83.5,   91.3,   92.5,   94.0,   94.9,   95.8,   96.7],
        "val_loss":   [1.4951, 0.7186, 0.5243, 0.4653, 0.4127, 0.4023, 0.4077, 0.3915, 0.3956],
        "val_top1":   [51.18,  75.00,  79.12,  83.24,  84.71,  85.59,  84.41,  85.88,  85.29],
        "val_top5":   [86.47,  96.76,  97.94,  98.53,  97.65,  98.24,  98.53,  98.82,  98.53],
        "val_binary": [88.82,  96.76,  96.47,  96.76,  96.76,  96.18,  96.47,  96.18,  96.47]
    }
    print("[OK] Đã khôi phục dữ liệu lịch sử 9 Epochs vào biến `history` để phục vụ vẽ biểu đồ ở Cell 7.")

# 4. HÀM ĐÁNH GIÁ TẬP TEST VỚI MULTI-CLIP + FLIP TTA (ĐÃ FIX MEMORY ALIGNMENT)
@torch.no_grad()
def evaluate_test_set_with_tta(
    eval_model: nn.Module,
    loader: DataLoader,
    criterion_fn,
    use_flip_tta: bool = True
) -> Dict:
    eval_model.eval()
    total_loss = 0.0
    all_probs, all_labels, all_violent = [], [], []

    pbar = tqdm(loader, desc="Đang đánh giá trên tập Test (Single-GPU)", leave=True)
    for clips, labels, is_violent in pbar:
        B, N_clips, T_f, C, H, W = clips.shape
        flat_clips = clips.reshape(B * N_clips, T_f, C, H, W).to(device, non_blocking=True).contiguous()
        labels_dev = labels.to(device, non_blocking=True)
        violent_dev = is_violent.to(device, non_blocking=True)

        with torch.amp.autocast("cuda", dtype=amp_dtype, enabled=cfg.use_amp):
            outputs_orig = eval_model(pixel_values=flat_clips, return_dict=False)
            logits_clips = outputs_orig[0].view(B, N_clips, -1)

            if use_flip_tta:
                flipped_clips = torch.flip(flat_clips, dims=[-1]).contiguous()
                outputs_flip = eval_model(pixel_values=flipped_clips, return_dict=False)
                logits_flip = outputs_flip[0].view(B, N_clips, -1)
                logits_clips = 0.5 * (logits_clips + logits_flip)
                del flipped_clips, outputs_flip

            lse_norm = torch.logsumexp(logits_clips, dim=1) - math.log(N_clips)
            video_logits = 0.5 * logits_clips.mean(dim=1) + 0.5 * lse_norm
            loss = criterion_fn(video_logits, labels_dev, violent_dev)

        total_loss += loss.item() * B
        probs = F.softmax(video_logits.float(), dim=-1).cpu().numpy()
        all_probs.append(probs)
        all_labels.append(labels.numpy())
        all_violent.append(is_violent.numpy())

        del flat_clips, outputs_orig, logits_clips, video_logits, loss

    all_probs = np.concatenate(all_probs, axis=0)
    all_labels = np.concatenate(all_labels, axis=0)
    all_violent = np.concatenate(all_violent, axis=0)
    all_preds = all_probs.argmax(axis=-1)

    top1_acc = top_k_accuracy_score(all_labels, all_probs, k=1, labels=np.arange(cfg.num_classes)) * 100.0
    if cfg.num_classes >= 5:
        top5_acc = top_k_accuracy_score(all_labels, all_probs, k=5, labels=np.arange(cfg.num_classes)) * 100.0
    else:
        top5_acc = top1_acc
    macro_f1 = f1_score(all_labels, all_preds, average="macro", zero_division=0) * 100.0
    weighted_f1 = f1_score(all_labels, all_preds, average="weighted", zero_division=0) * 100.0

    pred_violent = (all_preds < NUM_VIOLENCE_CLASSES).astype(int)
    bin_acc = accuracy_score(all_violent, pred_violent) * 100.0
    bin_prec = precision_score(all_violent, pred_violent, zero_division=0) * 100.0
    bin_rec = recall_score(all_violent, pred_violent, zero_division=0) * 100.0
    bin_f1 = f1_score(all_violent, pred_violent, zero_division=0) * 100.0

    return {
        "loss": total_loss / len(loader.dataset),
        "top1": top1_acc,
        "top5": top5_acc,
        "macro_f1": macro_f1,
        "weighted_f1": weighted_f1,
        "binary_acc": bin_acc,
        "binary_precision": bin_prec,
        "binary_recall": bin_rec,
        "binary_f1": bin_f1,
        "probs": all_probs,
        "labels": all_labels,
        "preds": all_preds,
        "is_violent": all_violent,
        "pred_violent": pred_violent
    }

# 5. THỰC THI ĐÁNH GIÁ TRÊN TẬP TEST
tta_views = cfg.test_num_clips * 2
t0_test = time.time()
test_metrics = evaluate_test_set_with_tta(raw_eval_model, test_loader, criterion, use_flip_tta=True)
test_time = time.time() - t0_test

test_probs = test_metrics["probs"]
test_labels = test_metrics["labels"]
test_preds = test_metrics["preds"]

print("\n" + "=" * 85)
print(f"KẾT QUẢ ĐÁNH GIÁ TRÊN TẬP TEST ({len(test_df)} VIDEOS | {cfg.test_num_clips} CLIPS x 2 TTA = {tta_views} VIEWS/VIDEO)")
print("=" * 85)
print(f"  1. HIỆU NĂNG PHÂN LOẠI {cfg.num_classes} LỚP HÀNH VI (MULTI-CLASS):")
print(f"     - Test Hierarchical Loss     : {test_metrics['loss']:.4f}")
print(f"     - Multi-class Top-1 Accuracy : {test_metrics['top1']:.2f}%")
print(f"     - Multi-class Top-5 Accuracy : {test_metrics['top5']:.2f}%")
print(f"     - Macro F1-Score             : {test_metrics['macro_f1']:.2f}%")
print(f"     - Weighted F1-Score          : {test_metrics['weighted_f1']:.2f}%")
print("-" * 85)
print(f"  2. HIỆU NĂNG PHÁT HIỆN BẠO LỰC NHỊ PHÂN (VIOLENCE vs NON-VIOLENCE):")
print(f"     - Binary Violence Accuracy   : {test_metrics['binary_acc']:.2f}%")
print(f"     - Binary Violence Precision  : {test_metrics['binary_precision']:.2f}%")
print(f"     - Binary Violence Recall     : {test_metrics['binary_recall']:.2f}%")
print(f"     - Binary Violence F1-Score   : {test_metrics['binary_f1']:.2f}%")
print("-" * 85)
print(f"  * Tổng thời gian suy luận Test  : {test_time:.1f}s ({test_time / len(test_df):.2f}s / video)")
print("=" * 85)

In [ ]:
# ==============================================================================
# CELL 6.5: BÁO CÁO PHÂN LOẠI CHI TIẾT (CLASSIFICATION REPORT CHUẨN SKLEARN)
# ==============================================================================
from sklearn.metrics import classification_report

# Lấy danh sách tên các lớp theo đúng thứ tự ID từ 0 đến num_classes - 1
target_names_list = [ID2LABEL[i] for i in range(cfg.num_classes)]

# Tạo báo cáo chuẩn sklearn
sklearn_report_str = classification_report(
    test_labels,
    test_preds,
    labels=np.arange(cfg.num_classes),
    target_names=target_names_list,
    digits=4,
    zero_division=0
)

# In kết quả ra màn hình với đường viền phân cách gọn gàng
print("=" * 115)
print(f"BÁO CÁO PHÂN LOẠI CHI TIẾT TRÊN TẬP TEST ({len(test_labels)} VIDEOS):")
print("=" * 115)
print(sklearn_report_str)
print("=" * 115)

In [ ]:
# ==============================================================================
# CELL 7: TRỰC QUAN HÓA TOÀN DIỆN (TRAINING CURVES, CONFUSION MATRICES & CLASS ACCURACY)
# ==============================================================================
plt.rcParams["figure.dpi"] = 120
plt.rcParams["savefig.dpi"] = 300
plt.rcParams["font.sans-serif"] = "DejaVu Sans"

# Danh sách tên lớp rút gọn nhẹ để hiển thị trên trục đồ thị không bị tràn lề
short_class_names = [
    f"{i:02d}. {ID2LABEL[i][:26]}..." if len(ID2LABEL[i]) > 28 else f"{i:02d}. {ID2LABEL[i]}"
    for i in range(cfg.num_classes)
]

# Nhãn dynamic cho binary confusion matrix
nv = NUM_VIOLENCE_CLASSES
bin_labels_display = [
    f"Violence (00..{nv-1:02d})",
    f"Non-Violence ({nv:02d}..{cfg.num_classes-1:02d})"
]

# ------------------------------------------------------------------------------
# 1. BIỂU ĐỒ ĐƯỜNG CONG HUẤN LUYỆN (LOSS & ACCURACY + VỊ TRÍ BEST EPOCH)
# ------------------------------------------------------------------------------
epochs_ran = len(history["train_loss"])
ep_axis = np.arange(1, epochs_ran + 1)

# Nếu chạy nhiều epoch, chỉ hiển thị tick cách đều để tránh chồng nhau
if epochs_ran > 15:
    tick_step = max(1, epochs_ran // 10)
    ep_ticks = np.arange(1, epochs_ran + 1, tick_step)
    if epochs_ran not in ep_ticks:
        ep_ticks = np.append(ep_ticks, epochs_ran)
else:
    ep_ticks = ep_axis

fig, axes = plt.subplots(1, 2, figsize=(16, 5.5))

# Subplot 1: Hierarchical Loss
axes[0].plot(ep_axis, history["train_loss"], label="Train Loss", marker="o", linewidth=2, color="#1f77b4")
axes[0].plot(ep_axis, history["val_loss"], label="Val Loss", marker="s", linewidth=2, color="#ff7f0e")
if best_epoch > 0:
    axes[0].axvline(best_epoch, color="crimson", linestyle="--", alpha=0.8, label=f"Best Epoch ({best_epoch})")
axes[0].set_title("Quá trình hội tụ Hàm mất mát (Hierarchical Loss)", fontsize=13, fontweight="bold")
axes[0].set_xlabel("Epoch", fontsize=11)
axes[0].set_ylabel("Loss", fontsize=11)
axes[0].set_xticks(ep_ticks)
axes[0].legend(frameon=True)
axes[0].grid(True, linestyle="--", alpha=0.4)

# Subplot 2: Top-1, Top-5 & Binary Accuracy
axes[1].plot(ep_axis, history["train_top1"], label="Train Top-1", linestyle="--", color="#7f7f7f", alpha=0.85)
axes[1].plot(ep_axis, history["val_top1"], label=f"Val Top-1 ({cfg.num_classes} lớp)", marker="o", linewidth=2.2, color="#2ca02c")
axes[1].plot(ep_axis, history["val_top5"], label=f"Val Top-5 ({cfg.num_classes} lớp)", marker="^", linewidth=1.8, color="#9467bd")
axes[1].plot(ep_axis, history["val_binary"], label="Val Binary (Violence vs Non-Violence)", marker="d", linewidth=1.8, color="#d62728")
if best_epoch > 0:
    axes[1].axvline(best_epoch, color="crimson", linestyle="--", alpha=0.8, label=f"Best Val Top-1 ({best_val_top1:.2f}%)")
axes[1].set_title("Độ chính xác Validation qua các Epoch", fontsize=13, fontweight="bold")
axes[1].set_xlabel("Epoch", fontsize=11)
axes[1].set_ylabel("Accuracy (%)", fontsize=11)
axes[1].set_xticks(ep_ticks)
axes[1].legend(loc="lower right", frameon=True)
axes[1].grid(True, linestyle="--", alpha=0.4)

plt.tight_layout()
curves_path = os.path.join(cfg.working_dir, "fig1_training_convergence_curves.png")
plt.savefig(curves_path, bbox_inches="tight")
plt.show()
plt.close(fig)

# ------------------------------------------------------------------------------
# 2. MA TRẬN NHẦM LẪN MULTI-CLASS (SỐ LƯỢNG MẪU & PHẦN TRĂM CHUẨN HÓA THEO HÀNG)
# ------------------------------------------------------------------------------
cm_counts = confusion_matrix(test_labels, test_preds, labels=np.arange(cfg.num_classes))
cm_norm = cm_counts.astype("float") / np.maximum(cm_counts.sum(axis=1, keepdims=True), 1) * 100.0

fig, axes = plt.subplots(1, 2, figsize=(22, 9.5))

# Bản 1: Số lượng video thực tế (Raw Counts)
sns.heatmap(
    cm_counts, annot=True, fmt="d", cmap="Blues", ax=axes[0],
    xticklabels=short_class_names, yticklabels=short_class_names,
    cbar_kws={"shrink": 0.8}, annot_kws={"size": 8.5}
)
axes[0].axhline(nv, color="crimson", linestyle="--", linewidth=2)
axes[0].axvline(nv, color="crimson", linestyle="--", linewidth=2)
axes[0].set_title(f"Confusion Matrix {cfg.num_classes} Lớp (Số lượng Video)", fontsize=13, fontweight="bold")
axes[0].set_xlabel("Nhãn Dự Đoán (Predicted)", fontsize=11)
axes[0].set_ylabel("Nhãn Thực Tế (Ground Truth)", fontsize=11)
axes[0].tick_params(axis="x", rotation=45)
axes[0].tick_params(axis="y", rotation=0)

# Bản 2: Chuẩn hóa theo phần trăm từng lớp (Row-Normalized Recall %)
sns.heatmap(
    cm_norm, annot=True, fmt=".1f", cmap="YlGnBu", ax=axes[1],
    xticklabels=short_class_names, yticklabels=False,
    cbar_kws={"shrink": 0.8, "label": "Recall (%)"}, annot_kws={"size": 8}
)
axes[1].axhline(nv, color="crimson", linestyle="--", linewidth=2)
axes[1].axvline(nv, color="crimson", linestyle="--", linewidth=2)
axes[1].set_title(f"Confusion Matrix {cfg.num_classes} Lớp Chuẩn hóa theo hàng (%)", fontsize=13, fontweight="bold")
axes[1].set_xlabel("Nhãn Dự Đoán (Predicted)", fontsize=11)
axes[1].set_ylabel("")
axes[1].tick_params(axis="x", rotation=45)

plt.suptitle(
    f"Phân tích Nhầm lẫn trên Tập Test "
    f"(Đường đỏ tách biệt {nv} lớp VIOLENCE [00..{nv-1:02d}] và {cfg.num_classes - nv} lớp NON-VIOLENCE [{nv:02d}..{cfg.num_classes-1:02d}])",
    fontsize=14, fontweight="bold", y=1.02
)
plt.tight_layout()
cm16_path = os.path.join(cfg.working_dir, "fig2_confusion_matrix_multiclass.png")
plt.savefig(cm16_path, bbox_inches="tight")
plt.show()
plt.close(fig)

# ------------------------------------------------------------------------------
# 3. MA TRẬN NHẦM LẪN NHỊ PHÂN & BIỂU ĐỒ TOP-1 / TOP-5 THEO TỪNG LỚP
# ------------------------------------------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(17, 6.5), gridspec_kw={"width_ratios": [1, 1.8]})

# Subplot Trái: Confusion Matrix Nhị phân (Non-Violence vs Violence)
cm_bin = confusion_matrix(test_metrics["is_violent"], test_metrics["pred_violent"], labels=[1, 0])
cm_bin_pct = cm_bin / np.maximum(cm_bin.sum(axis=1, keepdims=True), 1) * 100.0
bin_annot = np.array([
    [f"{cm_bin[i, j]}\n({cm_bin_pct[i, j]:.1f}%)" for j in range(2)]
    for i in range(2)
])
sns.heatmap(
    cm_bin, annot=bin_annot, fmt="", cmap="OrRd", ax=axes[0],
    xticklabels=bin_labels_display,
    yticklabels=bin_labels_display,
    cbar=False, annot_kws={"size": 12, "weight": "bold"}
)
axes[0].set_title(f"Ma trận Nhị phân (Binary Acc: {test_metrics['binary_acc']:.2f}%)", fontsize=13, fontweight="bold")
axes[0].set_xlabel("Nhóm Dự Đoán", fontsize=11)
axes[0].set_ylabel("Nhóm Thực Tế", fontsize=11)

# Subplot Phải: Biểu đồ ngang so sánh Top-1 & Top-5 của từng lớp
top5_k = min(5, cfg.num_classes)
top5_preds_mat = np.argsort(test_probs, axis=-1)[:, -top5_k:]
cls_top1_list, cls_top5_list = [], []
for c in range(cfg.num_classes):
    mask = (test_labels == c)
    cls_top1_list.append((test_preds[mask] == c).mean() * 100.0 if mask.sum() > 0 else 0.0)
    cls_top5_list.append(np.any(top5_preds_mat[mask] == c, axis=-1).mean() * 100.0 if mask.sum() > 0 else 0.0)

y_pos = np.arange(cfg.num_classes)
axes[1].barh(y_pos - 0.18, cls_top1_list, height=0.36, label="Top-1 Accuracy (%)", color="#2b5c8f")
axes[1].barh(y_pos + 0.18, cls_top5_list, height=0.36, label=f"Top-{top5_k} Accuracy (%)", color="#7fc97f", alpha=0.85)
axes[1].axhline(nv - 0.5, color="crimson", linestyle="--", linewidth=1.5, label="Ranh giới Violence / Non-Violence")
axes[1].set_yticks(y_pos)
axes[1].set_yticklabels(short_class_names, fontsize=9)
axes[1].invert_yaxis()
axes[1].set_xlim(0, 108)
axes[1].set_xlabel("Accuracy (%)", fontsize=11)
axes[1].set_title("Độ chính xác Top-1 & Top-5 chi tiết theo từng lớp trên Tập Test", fontsize=13, fontweight="bold")
axes[1].legend(loc="lower right")
axes[1].grid(True, axis="x", linestyle="--", alpha=0.4)

plt.tight_layout()
cm_bin_path = os.path.join(cfg.working_dir, "fig3_binary_cm_and_per_class_accuracy.png")
plt.savefig(cm_bin_path, bbox_inches="tight")
plt.show()
plt.close(fig)

print(f"[OK] Đã xuất 3 file hình ảnh chất lượng cao (300 DPI) tại thư mục: {cfg.working_dir}")

In [ ]:
# ==============================================================================
# CELL 8: BÁO CÁO KẾT QUẢ CHI TIẾT TOP-1 & TOP-5 TEST + PHÂN TÍCH LỖI NHẦM LẪN
# ==============================================================================
nv = NUM_VIOLENCE_CLASSES
top5_k = min(5, cfg.num_classes)
top5_preds_matrix = np.argsort(test_probs, axis=-1)[:, -top5_k:]

prec_arr, rec_arr, f1_arr, support_arr = precision_recall_fscore_support(
    test_labels, test_preds, labels=np.arange(cfg.num_classes), zero_division=0
)

# 1. Thống kê chi tiết Top-1, Top-5, Precision, Recall, F1 và Cross-Group Error cho từng lớp
class_report_rows = []
for cls_id in range(cfg.num_classes):
    mask = (test_labels == cls_id)
    n_support = int(mask.sum())
    is_vio_cls = (cls_id < nv)

    if n_support > 0:
        c_top1 = (test_preds[mask] == cls_id).mean() * 100.0
        c_top5 = np.any(top5_preds_matrix[mask] == cls_id, axis=-1).mean() * 100.0
        if is_vio_cls:
            group_keep_rate = (test_preds[mask] < nv).mean() * 100.0
        else:
            group_keep_rate = (test_preds[mask] >= nv).mean() * 100.0
    else:
        c_top1, c_top5, group_keep_rate = 0.0, 0.0, 0.0

    class_report_rows.append({
        "ID": f"{cls_id:02d}",
        "Nhóm": "Violence" if is_vio_cls else "Non-Violence",
        "Tên Lớp": ID2LABEL[cls_id],
        "Số mẫu": n_support,
        "Top-1 (%)": round(c_top1, 2),
        f"Top-{top5_k} (%)": round(c_top5, 2),
        "Đúng Nhóm (%)": round(group_keep_rate, 2),
        "Precision (%)": round(prec_arr[cls_id] * 100.0, 2),
        "Recall (%)": round(rec_arr[cls_id] * 100.0, 2),
        "F1-Score (%)": round(f1_arr[cls_id] * 100.0, 2),
    })

report_df = pd.DataFrame(class_report_rows)

# 2. Tính chỉ số tổng hợp theo nhóm Violence vs Non-Violence và Toàn cục
vio_df = report_df.iloc[:nv]
non_vio_df = report_df.iloc[nv:]

summary_rows = [
    {
        "Phân Nhóm Đánh Giá": f"1. Nhóm VIOLENCE ({nv} lớp: 00..{nv-1:02d})",
        "Số mẫu": int(vio_df["Số mẫu"].sum()),
        "Macro Top-1 (%)": round(vio_df["Top-1 (%)"].mean(), 2),
        f"Macro Top-{top5_k} (%)": round(vio_df[f"Top-{top5_k} (%)"].mean(), 2),
        "Đúng Nhóm (%)": round(vio_df["Đúng Nhóm (%)"].mean(), 2),
        "Macro F1 (%)": round(vio_df["F1-Score (%)"].mean(), 2),
    },
    {
        "Phân Nhóm Đánh Giá": f"2. Nhóm NON-VIOLENCE ({cfg.num_classes - nv} lớp: {nv:02d}..{cfg.num_classes-1:02d})",
        "Số mẫu": int(non_vio_df["Số mẫu"].sum()),
        "Macro Top-1 (%)": round(non_vio_df["Top-1 (%)"].mean(), 2),
        f"Macro Top-{top5_k} (%)": round(non_vio_df[f"Top-{top5_k} (%)"].mean(), 2),
        "Đúng Nhóm (%)": round(non_vio_df["Đúng Nhóm (%)"].mean(), 2),
        "Macro F1 (%)": round(non_vio_df["F1-Score (%)"].mean(), 2),
    },
    {
        "Phân Nhóm Đánh Giá": f"3. TOÀN TẬP TEST (Macro Average {cfg.num_classes} lớp)",
        "Số mẫu": int(report_df["Số mẫu"].sum()),
        "Macro Top-1 (%)": round(report_df["Top-1 (%)"].mean(), 2),
        f"Macro Top-{top5_k} (%)": round(report_df[f"Top-{top5_k} (%)"].mean(), 2),
        "Đúng Nhóm (%)": round(report_df["Đúng Nhóm (%)"].mean(), 2),
        "Macro F1 (%)": round(report_df["F1-Score (%)"].mean(), 2),
    },
    {
        "Phân Nhóm Đánh Giá": "4. TOÀN TẬP TEST (Micro / Overall)",
        "Số mẫu": len(test_labels),
        "Micro Top-1 (%)": round(test_metrics["top1"], 2),
        f"Micro Top-{top5_k} (%)": round(test_metrics["top5"], 2),
        "Đúng Nhóm (%)": round(test_metrics["binary_acc"], 2),
        "Weighted F1 (%)": round(test_metrics["weighted_f1"], 2),
    }
]
group_summary_df = pd.DataFrame(summary_rows)

# 3. Phân tích Top các cặp lớp bị nhầm lẫn nhiều nhất (Confusion Pairs Analysis)
cm_matrix = confusion_matrix(test_labels, test_preds, labels=np.arange(cfg.num_classes))
confused_pairs = []
for i in range(cfg.num_classes):
    for j in range(cfg.num_classes):
        if i != j and cm_matrix[i, j] > 0:
            same_group = (i < nv) == (j < nv)
            error_type = (
                "Nội bộ nhóm (Ít nghiêm trọng)" if same_group
                else ("Bỏ lọt Bạo lực [FN - Nguy hiểm]" if i < nv else "Báo động giả [FP]")
            )
            confused_pairs.append({
                "Nhãn Thực Tế (True)": f"{i:02d}. {ID2LABEL[i]}",
                "Nhãn Dự Đoán Nhầm (Pred)": f"{j:02d}. {ID2LABEL[j]}",
                "Số Video Nhầm": int(cm_matrix[i, j]),
                "Tỷ lệ / Lớp Thực Tế (%)": round(cm_matrix[i, j] / max(1, cm_matrix[i].sum()) * 100.0, 2),
                "Loại Lỗi": error_type
            })

confused_df = pd.DataFrame(confused_pairs).sort_values(by="Số Video Nhầm", ascending=False).reset_index(drop=True)

# 4. Lưu toàn bộ báo cáo ra CSV trong /kaggle/working
detail_csv = os.path.join(cfg.working_dir, f"report_{cfg.num_classes}classes_top1_top{top5_k}_detailed.csv")
summary_csv = os.path.join(cfg.working_dir, "report_group_summary.csv")
preds_csv = os.path.join(cfg.working_dir, "test_video_predictions_full.csv")

report_df.to_csv(detail_csv, index=False)
group_summary_df.to_csv(summary_csv, index=False)

# Lưu thêm bảng dự đoán chi tiết từng video trong tập Test
test_pred_log = test_df.copy()
test_pred_log["pred_label"] = test_preds
test_pred_log["pred_class_name"] = [ID2LABEL[p] for p in test_preds]
test_pred_log["pred_confidence (%)"] = np.round(test_probs.max(axis=-1) * 100.0, 2)
test_pred_log["top1_correct"] = (test_preds == test_labels)
test_pred_log["top5_correct"] = [
    bool(lbl in top5_row) for lbl, top5_row in zip(test_labels, top5_preds_matrix)
]
test_pred_log.to_csv(preds_csv, index=False)

# 5. In Báo cáo Tổng hợp ra màn hình
print("=" * 115)
print(f"BẢNG 1: BÁO CÁO KẾT QUẢ TOP-1, TOP-{top5_k}, ĐỘ CHÍNH XÁC NHÓM & F1-SCORE TRÊN TỪNG LỚP (TẬP TEST - {len(test_labels)} VIDEOS)")
print("=" * 115)
print(report_df.to_string(index=False))

print("\n" + "=" * 115)
print("BẢNG 2: TỔNG HỢP HIỆU NĂNG THEO NHÓM (VIOLENCE vs NON-VIOLENCE) & TOÀN CỤC")
print("=" * 115)
print(group_summary_df.to_string(index=False))

print("\n" + "=" * 115)
print("BẢNG 3: TOP 10 CẶP HÀNH VI BỊ NHẦM LẪN NHIỀU NHẤT TRÊN TẬP TEST")
print("=" * 115)
if len(confused_df) > 0:
    print(confused_df.head(10).to_string(index=False))
else:
    print("  Tuyệt vời! Không có video nào bị dự đoán sai trên tập Test.")
print("=" * 115)
print("ĐÃ XUẤT THÀNH CÔNG CÁC FILE BÁO CÁO:")
print(f"  1. Báo cáo chi tiết {cfg.num_classes} lớp : {detail_csv}")
print(f"  2. Báo cáo tổng hợp nhóm    : {summary_csv}")
print(f"  3. Log dự đoán từng video   : {preds_csv}")
print("=" * 115)